# 02 LangGraph STORM 長文寫作系統

這一章把 LCEL 直線流程升級成 LangGraph 狀態圖。STORM 的「多視角研究 → 大綱 → 寫作 → 品質檢查」會變成明確節點，方便觀察、測試與擴充。

## 學習目標

- 使用 `StateGraph` 建立長文寫作流程。
- 用 state 保存 topic、research_notes、outline、draft、review。
- 理解節點如何取代舊式「多智能體框架」黑箱。
- 為下一章 deep research fan-out/fan-in 做準備。

In [ ]:
import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph

load_dotenv()
MODEL_NAME = os.getenv("COURSE_MODEL", "openai:gpt-4o-mini")
model = init_chat_model(MODEL_NAME)
print(f"使用模型: {MODEL_NAME}")

## 1. 定義狀態

LangGraph 的核心是 state。每個節點只讀需要的欄位，回傳要更新的欄位。

In [ ]:
class WritingState(TypedDict, total=False):
    topic: str
    audience: str
    research_notes: str
    outline: str
    draft: str
    review: str
    final_article: str

## 2. 定義節點

In [ ]:
def research_node(state: WritingState) -> dict:
    prompt = f"""你是研究編輯。
主題：{state["topic"]}
目標讀者：{state.get("audience", "一般讀者")}

請整理：背景、主要觀點、案例、風險、可延伸查證的方向。"""
    response = model.invoke([HumanMessage(content=prompt)])
    return {"research_notes": str(response.content)}


def outline_node(state: WritingState) -> dict:
    prompt = f"""你是主編。根據研究摘要設計長文大綱。
主題：{state["topic"]}
研究摘要：
{state["research_notes"]}

請輸出 Markdown 大綱。"""
    response = model.invoke([HumanMessage(content=prompt)])
    return {"outline": str(response.content)}


def draft_node(state: WritingState) -> dict:
    prompt = f"""你是長文作者。請根據研究摘要與大綱撰寫初稿。
目標讀者：{state.get("audience", "一般讀者")}

研究摘要：
{state["research_notes"]}

大綱：
{state["outline"]}
"""
    response = model.invoke([HumanMessage(content=prompt)])
    return {"draft": str(response.content)}


def review_node(state: WritingState) -> dict:
    prompt = f"""你是嚴格編輯。檢查以下文章：

{state["draft"]}

請指出：結構問題、事實風險、讀者理解障礙、可以改善的段落。"""
    response = model.invoke([HumanMessage(content=prompt)])
    return {"review": str(response.content)}


def revise_node(state: WritingState) -> dict:
    prompt = f"""你是作者。根據編輯意見修訂文章。

初稿：
{state["draft"]}

編輯意見：
{state["review"]}

請輸出 final article。"""
    response = model.invoke([HumanMessage(content=prompt)])
    return {"final_article": str(response.content)}

## 3. 組裝 Graph

In [ ]:
builder = StateGraph(WritingState)
builder.add_node("research", research_node)
builder.add_node("outline", outline_node)
builder.add_node("draft", draft_node)
builder.add_node("review", review_node)
builder.add_node("revise", revise_node)

builder.add_edge(START, "research")
builder.add_edge("research", "outline")
builder.add_edge("outline", "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", "revise")
builder.add_edge("revise", END)

writing_graph = builder.compile()
print("✅ LangGraph STORM 寫作流程建立完成")

## 4. 執行流程

In [ ]:
# result = writing_graph.invoke({
#     "topic": "AI 對台灣中小企業的影響",
#     "audience": "企業主管",
# })
# print(result["final_article"][:2000])

## 5. 為什麼 LangGraph 比框架黑箱更適合教學？

- 每個節點都能單獨測試。
- state 讓資料流很清楚。
- 未來可以加條件路由：如果 review 不合格就回到 draft。
- 未來可以加 human-in-the-loop：老師或編輯先審大綱再寫。
- 下一章可以把 research 節點替換成 deep research 子圖。